# DSP Allocation in Quantized CNN Accelerators
### HLS Estimates Against Synthesized Netlists

Reproduction notebook for the IEEE Embedded Systems Letters manuscript by
M. Tasci and A. Akkaya, Bandirma Onyedi Eylul University.

Repository: https://github.com/mtasci42/hls-dsp-allocation

This notebook covers everything that runs on a host machine:

| Section | Produces |
|---|---|
| 1. Configuration | experiment matrix, directory layout |
| 2. Dataset | CWRU leave-one-load-out split |
| 3. Float baseline | trained FP32 model with batch normalization folded in |
| 4. Quantizer machinery | weight and activation quantizers, quantized model builder |
| 5. Activation range calibration | the clipping percentile, selected on validation |
| 6. Quantization-aware fine-tuning | the locked accuracy table, deployed weights |
| 7. Pure-integer golden reference | bit-exact integer inference, accumulator widths |
| 8. HLS artifacts | `weights.h`, `test_data.h`, C++ sources, Tcl scripts |
| 9. Board export | `golden_<config>.npz` for the PYNQ-Z1 |
| 10. Figures | the manuscript figures |
| Appendix | two diagnostics that justify design decisions |

Synthesis, implementation and board measurement are **not** run here. Those steps
use the Tcl scripts emitted in Section 8 and the measurement notebook
`02_pynq_measure.ipynb`.

Runtime on a Colab T4: about 25 minutes end to end, dominated by Section 6.

## 1. Configuration

Everything downstream is driven by the `EXPERIMENTS` dictionary defined here.

Five configurations are studied. Only the **second convolution** is swept: it holds
92% of the weights and the entire multiplier array. The first convolution and the
dense classifier keep eight-bit weights throughout, the former because it sees the
raw signal through a single input channel and is the accuracy bottleneck, the
latter because it costs three DSP blocks. Configuration names therefore refer to
the second convolution.

| Name | conv2 weights | activations | role |
|---|---|---|---|
| `W_int8_A8` | int8 | 8 bit | baseline |
| `W_int4_A8` | int4 | 8 bit | narrow weight operand |
| `A_int8_A4` | int8 | 4 bit | narrow activation operand |
| `A_int4_A4` | int4 | 4 bit | both narrowed |
| `W_ternary_A8` | ternary | 8 bit | multiplier-free |

`W_int4_A8` and `A_int8_A4` are the controlled pair: identical 19-bit accumulators,
opposite narrowed operands.

The input quantizer `qin` is fixed at 8 bits signed in every configuration, so the
AXI transfer carries one byte per sample and memory traffic is identical throughout.

`ROOT` below is a plain relative path. On Colab with Drive mounted, set it to the
absolute path of your working folder instead.

In [ ]:
# ========================= CENTRAL CONFIGURATION =========================
import os, urllib.request, numpy as np, random

SEED = 42
os.environ["PYTHONHASHSEED"]   = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED); np.random.seed(SEED)

ROOT = "./rulman_v2"          # on Colab: "/content/drive/MyDrive/<your folder>/rulman_v2"
DIRS = {k: os.path.join(ROOT, k) for k in ["dataset", "model", "output", "fig", "hls"]}
RAW_DIR = os.path.join(DIRS["dataset"], "raw")
for d in list(DIRS.values()) + [RAW_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- data source ----
RAW_BASE  = "https://raw.githubusercontent.com/srigas/CWRU_Bearing_NumPy/main/Data"
CHANNEL   = "DE"
HP_TO_RPM = {0: "1797", 1: "1772", 2: "1750", 3: "1730"}

CLASSES = ["Normal","IR_7","IR_14","IR_21","B_7","B_14","B_21","OR_7","OR_14","OR_21"]
def class_files(rpm):
    d = {"Normal": [f"{rpm}_Normal.npz"]}
    for s in ["7", "14", "21"]:
        d[f"IR_{s}"] = [f"{rpm}_IR_{s}_DE12.npz"]
        d[f"B_{s}"]  = [f"{rpm}_B_{s}_DE12.npz"]
        d[f"OR_{s}"] = [f"{rpm}_OR@6_{s}_DE12.npz"]
    return d
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES  = len(CLASSES)

WIN, STRIDE = 2048, 1024
TRAIN_HP, VAL_HP, TEST_HP = [1, 2], [0], [3]
CACHE = os.path.join(DIRS["dataset"], "cwru_lolo.npz")

# ---- architecture, fixed across the study ----
ARCH = dict(c1_out=16, c1_k=7, c2_out=32, c2_k=5, stride=2)

# ========================= EXPERIMENT MATRIX =========================
# wbits : layer -> {"int8", "int4", "ternary"}
# abits : activation tensor -> number of bits. qin is always 8 (int8 AXI stream).
def mk(name, w2, a, w1="int8", wd="int8", axis="W"):
    return dict(name=name, axis=axis,
                wbits={"conv1": w1, "conv2": w2, "dense": wd},
                abits={"qin": 8, "qa1": a, "qa2": a})

EXPERIMENTS = {}
for w in ["int8", "int4", "ternary"]:                 # weight axis, 8-bit activations
    c = mk(f"W_{w}_A8", w, 8, axis="W"); EXPERIMENTS[c["name"]] = c
for w in ["int8", "int4"]:                            # activation axis
    c = mk(f"A_{w}_A4", w, 4, axis="A"); EXPERIMENTS[c["name"]] = c

BIT_COST = {"int8": 8, "int4": 4, "ternary": 2}
LAYER_NW = {"conv1": ARCH["c1_k"] * 1 * ARCH["c1_out"],
            "conv2": ARCH["c2_k"] * ARCH["c1_out"] * ARCH["c2_out"],
            "dense": ARCH["c2_out"] * NUM_CLASSES}
def weight_bits(wb):
    return sum(LAYER_NW[l] * BIT_COST[wb[l]] for l in LAYER_NW)

print(f"{NUM_CLASSES} classes | window {WIN}/{STRIDE} | "
      f"LOLO train {TRAIN_HP} -> val {VAL_HP} -> test {TEST_HP}")
print(f"weights per layer: {LAYER_NW} (total {sum(LAYER_NW.values())})")
print(f"\n{len(EXPERIMENTS)} configurations:")
for n, c in EXPERIMENTS.items():
    print(f"  [{c['axis']}] {n:14s} w={c['wbits']} a1/a2={c['abits']['qa1']} "
          f"| {weight_bits(c['wbits'])} weight bits")

## 2. Dataset

Case Western Reserve University drive-end accelerometer set, 12 kHz, ten classes:
normal plus inner-race, ball and outer-race faults at three severities. The raw
data are **not redistributed**; they are fetched from the `srigas/CWRU_Bearing_NumPy`
mirror at run time.

Windows of 2048 samples with a hop of 1024, standardized individually so that
amplitude differences between operating regimes cannot leak class information.

Leave-one-load-out split: each motor load goes to exactly one partition, so the
test regime is never seen during training. The assertions at the end of the cell
catch load leakage and missing classes.

Results are cached to `dataset/cwru_lolo.npz`; a second run skips the download.

In [ ]:
# ========================= DATA LOADING (leave-one-load-out) =========================
def _download(fname):
    rpm = fname.split("_")[0]; local = os.path.join(RAW_DIR, fname)
    if os.path.exists(local) and os.path.getsize(local) > 0: return local
    urllib.request.urlretrieve(f"{RAW_BASE}/{rpm}%20RPM/{urllib.request.quote(fname)}", local)
    return local

def _load_de(path):
    z = np.load(path)
    if CHANNEL not in z.files: raise KeyError(f"{path}: channel {CHANNEL} not in {z.files}")
    return z[CHANNEL].astype(np.float32).reshape(-1)

def _windows(sig):
    n = 1 + (len(sig) - WIN)//STRIDE if len(sig) >= WIN else 0
    idx = np.arange(WIN)[None,:] + STRIDE*np.arange(n)[:,None]
    return sig[idx] if n > 0 else np.empty((0, WIN), np.float32)

def _norm(W):
    m = W.mean(1, keepdims=True); s = W.std(1, keepdims=True) + 1e-8
    return (W - m)/s

if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    X_train, y_train, load_train = z["X_train"], z["y_train"], z["load_train"]
    X_val,   y_val,   load_val   = z["X_val"],   z["y_val"],   z["load_val"]
    X_test,  y_test,  load_test  = z["X_test"],  z["y_test"],  z["load_test"]
    print(f"loaded from cache: {CACHE}")
else:
    SPLIT_OF = {}
    for hp in TRAIN_HP: SPLIT_OF[hp] = "train"
    for hp in VAL_HP:   SPLIT_OF[hp] = "val"
    for hp in TEST_HP:  SPLIT_OF[hp] = "test"

    buf = {s: [[], [], []] for s in ["train","val","test"]}
    for hp in TRAIN_HP + VAL_HP + TEST_HP:
        sp = SPLIT_OF[hp]
        cf = class_files(HP_TO_RPM[hp])
        for cls, fns in cf.items():
            for fname in fns:
                W = _windows(_load_de(_download(fname)))
                if len(W) == 0: print(f"  WARNING: {fname} too short"); continue
                buf[sp][0].append(W)
                buf[sp][1] += [CLASS_TO_IDX[cls]]*len(W)
                buf[sp][2] += [hp]*len(W)

    def _stack(sp):
        Xs, ys, ls = buf[sp]
        return (_norm(np.concatenate(Xs, 0)).astype(np.float32)[..., None],
                np.array(ys, np.int64), np.array(ls, np.int64))
    X_train, y_train, load_train = _stack("train")
    X_val,   y_val,   load_val   = _stack("val")
    X_test,  y_test,  load_test  = _stack("test")

    np.savez_compressed(CACHE,
        X_train=X_train, y_train=y_train, load_train=load_train,
        X_val=X_val, y_val=y_val, load_val=load_val,
        X_test=X_test, y_test=y_test, load_test=load_test,
        classes=np.array(CLASSES), win=WIN, stride=STRIDE)
    print(f"cache written: {CACHE}")

for nm, y, l in [("TRAIN", y_train, load_train), ("VAL", y_val, load_val), ("TEST", y_test, load_test)]:
    print(f"[{nm}] {len(y)} windows | loads {sorted(set(l.tolist()))} | "
          f"{ {CLASSES[i]: int((y==i).sum()) for i in range(NUM_CLASSES)} }")

assert set(load_train.tolist()).isdisjoint(load_val.tolist())
assert set(load_train.tolist()).isdisjoint(load_test.tolist())
assert set(load_val.tolist()).isdisjoint(load_test.tolist())
for nm, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    assert set(y.tolist()) == set(range(NUM_CLASSES)), f"missing class in {nm}"
print(f"\ninput shape: {X_train.shape[1:]}")

## 3. Float baseline and batch-normalization folding

The classifier is a deliberately small 1-D CNN of about 3000 weights:
`Conv1D(16, k=7, s=2) - BN - ReLU -> Conv1D(32, k=5, s=2) - BN - ReLU -> GAP -> Dense(10)`.
Convolutions carry no bias, because batch normalization is folded into the weights
afterwards and the folded bias takes its place.

Folding is an affine transform per output channel:
`w_fold[o] = g[o]*w[o]/sqrt(var[o]+eps)`, `b_fold[o] = b[o] - g[o]*mu[o]/sqrt(var[o]+eps)`.

The assertion at the end verifies that the folded model matches the trained one
numerically. If it fails, nothing below is valid, so the cell stops there.

The float model is trained **once**, saved, and reloaded for every quantization
configuration, so all results share an initialization.

In [ ]:
# ========================= FLOAT TRAINING AND BN FOLDING =========================
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print("op-level determinism unavailable:", e)

FP32_PATH   = os.path.join(DIRS["model"], "fp32.keras")
FOLDED_PATH = os.path.join(DIRS["model"], "folded.weights.h5")

def build_fp32():
    inp = layers.Input((WIN, 1), name="sig")
    x = layers.Conv1D(ARCH["c1_out"], ARCH["c1_k"], strides=ARCH["stride"],
                      padding="same", use_bias=False, name="conv1")(inp)
    x = layers.BatchNormalization(name="bn1")(x); x = layers.ReLU(name="relu1")(x)
    x = layers.Conv1D(ARCH["c2_out"], ARCH["c2_k"], strides=ARCH["stride"],
                      padding="same", use_bias=False, name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x); x = layers.ReLU(name="relu2")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    out = layers.Dense(NUM_CLASSES, name="dense")(x)
    return models.Model(inp, out)

def build_folded():
    """Same architecture, no BN, convolutions carry the folded bias."""
    inp = layers.Input((WIN, 1), name="sig")
    x = layers.Conv1D(ARCH["c1_out"], ARCH["c1_k"], strides=ARCH["stride"],
                      padding="same", use_bias=True, name="conv1")(inp)
    x = layers.ReLU(name="relu1")(x)
    x = layers.Conv1D(ARCH["c2_out"], ARCH["c2_k"], strides=ARCH["stride"],
                      padding="same", use_bias=True, name="conv2")(x)
    x = layers.ReLU(name="relu2")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    out = layers.Dense(NUM_CLASSES, name="dense")(x)
    return models.Model(inp, out)

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}

def macro_f1(model, X, y):
    return f1_score(y, model.predict(X, verbose=0).argmax(1), average="macro")

if os.path.exists(FP32_PATH):
    fp32 = models.load_model(FP32_PATH); print("float model loaded from disk")
else:
    fp32 = build_fp32()
    fp32.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=["accuracy"])
    fp32.fit(X_train, y_train, validation_data=(X_val, y_val),
             epochs=60, batch_size=128, class_weight=CLASS_WEIGHT, verbose=2,
             callbacks=[tf.keras.callbacks.EarlyStopping(
                 monitor="val_loss", patience=8, restore_best_weights=True)])
    fp32.save(FP32_PATH)

F1_FP32 = macro_f1(fp32, X_test, y_test)
print(f"\nfloat cross-load test macro-F1 : {F1_FP32:.4f}")
print(f"float validation macro-F1      : {macro_f1(fp32, X_val, y_val):.4f}")

# ---- fold batch normalization ----
folded = build_folded()
for cn, bn in [("conv1", "bn1"), ("conv2", "bn2")]:
    W  = fp32.get_layer(cn).get_weights()[0]              # (k, Cin, Cout)
    g, b, mu, var = fp32.get_layer(bn).get_weights()
    eps = fp32.get_layer(bn).epsilon
    s = g/np.sqrt(var + eps)
    folded.get_layer(cn).set_weights([W*s[None, None, :], (b - mu*s).astype(np.float32)])
folded.get_layer("dense").set_weights(fp32.get_layer("dense").get_weights())

d = np.abs(fp32.predict(X_test[:512], verbose=0) - folded.predict(X_test[:512], verbose=0)).max()
F1_FOLD = macro_f1(folded, X_test, y_test)
print(f"folded  test macro-F1          : {F1_FOLD:.4f}")
print(f"max absolute logit difference  : {d:.3e}")
assert d < 1e-3, "BN folding does not match the trained model"
folded.save_weights(FOLDED_PATH)
print(f"folded weights -> {FOLDED_PATH}")

## 4. Quantizer machinery

`quant_weight(w, mode)` quantizes per output channel. For `int8` and `int4` the
scale is `max|w| / (2^(b-1) - 1)`. For `ternary` the codebook is `{-1, 0, +1}`
through the threshold rule `D = 0.7 * mean|w|` with scale `a = mean(|w| : |w| > D)`.

`QAct(bits, signed)` quantizes an activation tensor. The range is tracked by an
exponential moving average during training, but in this study it is **frozen** at a
fixed percentile of the float model's activation distribution (Section 5), so that
calibration is not a free variable.

Both use the straight-through estimator, `w + stop_gradient(q(w) - w)`, so the
forward pass sees the quantized value while the gradient flows through the identity.

The unit test at the end checks that each mode produces the expected number of
levels: 3 for ternary, at most 15 for int4.

In [ ]:
# ========================= QUANTIZERS (weights and parametric activations) =========================
import tensorflow as tf
from tensorflow.keras import layers

WBITS_N = {"int8": 8, "int4": 4, "int2": 2}

def quant_weight(w, mode):
    """Per-output-channel symmetric or ternary quantization with a straight-through estimator."""
    ra = list(range(len(w.shape) - 1))
    absw = tf.abs(w)
    if mode == "ternary":
        delta = 0.7 * tf.reduce_mean(absw, axis=ra, keepdims=True)
        mask  = tf.cast(absw > delta, w.dtype)
        alpha = tf.reduce_sum(absw*mask, axis=ra, keepdims=True) / \
                (tf.reduce_sum(mask, axis=ra, keepdims=True) + 1e-8)
        q = alpha * tf.sign(w) * mask
    else:
        n = 2**(WBITS_N[mode] - 1) - 1                    # int8 -> 127, int4 -> 7
        s = tf.reduce_max(absw, axis=ra, keepdims=True) / n + 1e-12
        q = tf.clip_by_value(tf.round(w/s), -n, n) * s
    return w + tf.stop_gradient(q - w)


class QAct(layers.Layer):
    """Per-tensor activation quantizer with a parametric bit-width."""
    def __init__(self, bits=8, signed=False, momentum=0.95, **kw):
        super().__init__(**kw)
        self.bits, self.signed, self.momentum = int(bits), bool(signed), momentum
    def build(self, _):
        self.rng    = self.add_weight(name="rng",    shape=(), initializer="zeros", trainable=False)
        self.inited = self.add_weight(name="inited", shape=(), initializer="zeros", trainable=False)
    def scale(self):
        n = (2**(self.bits-1) - 1) if self.signed else (2**self.bits - 1)
        return tf.maximum(self.rng, 1e-8) / n
    def call(self, x, training=None):
        if training:
            cur = tf.reduce_max(tf.abs(x)) if self.signed else tf.reduce_max(tf.maximum(x, 0.0))
            new = tf.where(self.inited > 0,
                           self.momentum*self.rng + (1-self.momentum)*cur, cur)
            self.rng.assign(new); self.inited.assign(1.0)
        s = self.scale()
        if self.signed:
            n = 2**(self.bits-1) - 1
            q = tf.clip_by_value(tf.round(x/s), -n, n) * s
        else:
            n = 2**self.bits - 1
            q = tf.clip_by_value(tf.round(x/s), 0, n) * s
        return x + tf.stop_gradient(q - x)
    def get_config(self):
        c = super().get_config()
        c.update(bits=self.bits, signed=self.signed, momentum=self.momentum); return c


class QConv1D(layers.Layer):
    def __init__(self, filters, k, stride, mode, **kw):
        super().__init__(**kw)
        self.filters, self.k, self.stride, self.mode = filters, k, stride, mode
    def build(self, ish):
        self.kernel = self.add_weight(shape=(self.k, int(ish[-1]), self.filters),
                                      initializer="glorot_uniform",
                                      trainable=True, name="kernel")
        self.bias   = self.add_weight(shape=(self.filters,),
                                      initializer="zeros",
                                      trainable=True, name="bias")
    def call(self, x):
        w = quant_weight(self.kernel, self.mode)
        y = tf.nn.conv1d(x, w, stride=self.stride, padding="SAME")
        return tf.nn.bias_add(y, self.bias)
    def get_config(self):
        c = super().get_config()
        c.update(filters=self.filters, k=self.k, stride=self.stride, mode=self.mode); return c


class QDense(layers.Layer):
    def __init__(self, units, mode, **kw):
        super().__init__(**kw); self.units, self.mode = units, mode
    def build(self, ish):
        self.kernel = self.add_weight(shape=(int(ish[-1]), self.units),
                                      initializer="glorot_uniform",
                                      trainable=True, name="kernel")
        self.bias   = self.add_weight(shape=(self.units,),
                                      initializer="zeros",
                                      trainable=True, name="bias")
    def call(self, x):
        return tf.matmul(x, quant_weight(self.kernel, self.mode)) + self.bias
    def get_config(self):
        c = super().get_config(); c.update(units=self.units, mode=self.mode); return c

# --- unit test: are the codebooks what we expect? ---
_w = tf.constant(np.random.randn(5, 16, 32).astype(np.float32))
for m in ["int8", "int4", "ternary"]:
    q = quant_weight(_w, m).numpy()
    lv = np.unique(np.round(q[:, :, 0] / (np.abs(q[:, :, 0]).max() + 1e-12), 6))
    print(f"{m:8s} distinct levels in channel 0: {len(lv):3d}")

### 4.1 Quantized model and wiring check

`build_quant(cfg)` assembles the quantized model from a configuration dictionary.
`load_folded_into` copies the folded float weights in, so every configuration starts
from the same point.

The check at the end is important: with all-int8 weights and 8-bit activations, and
without any fine-tuning, the model should score close to the folded float model. A
large gap means the quantizers are not wired into the forward pass.

In [ ]:
# ========================= QUANTIZED MODEL AND WIRING CHECK =========================
from tensorflow.keras import models

def build_quant(cfg):
    wb, ab = cfg["wbits"], cfg["abits"]
    inp = layers.Input((WIN, 1), name="sig")
    x = QAct(ab["qin"], signed=True,  name="qin")(inp)
    x = QConv1D(ARCH["c1_out"], ARCH["c1_k"], ARCH["stride"], wb["conv1"], name="conv1")(x)
    x = layers.ReLU(name="relu1")(x)
    x = QAct(ab["qa1"], signed=False, name="qa1")(x)
    x = QConv1D(ARCH["c2_out"], ARCH["c2_k"], ARCH["stride"], wb["conv2"], name="conv2")(x)
    x = layers.ReLU(name="relu2")(x)
    x = QAct(ab["qa2"], signed=False, name="qa2")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    out = QDense(NUM_CLASSES, wb["dense"], name="dense")(x)
    return models.Model(inp, out, name=cfg["name"])

def load_folded_into(qm):
    """Copy the folded float weights into a quantized model (identical initialization)."""
    ref = build_folded(); ref.load_weights(FOLDED_PATH)
    for ln in ["conv1", "conv2", "dense"]:
        qm.get_layer(ln).set_weights(ref.get_layer(ln).get_weights())
    return qm

def calibrate(qm, n_batch=20, bs=128):
    """Forward passes only: fills the activation EMA ranges."""
    idx = np.random.default_rng(SEED).choice(len(X_train), min(n_batch*bs, len(X_train)), replace=False)
    for i in range(0, len(idx), bs):
        qm(X_train[idx[i:i+bs]], training=True)
    return qm

def q_macro_f1(qm, X, y):
    return f1_score(y, qm.predict(X, verbose=0).argmax(1), average="macro")

# --- wiring test: W_int8_A8, post-training quantization only ---
cfg0 = EXPERIMENTS["W_int8_A8"]
qm0  = calibrate(load_folded_into(build_quant(cfg0)))
f1_ptq = q_macro_f1(qm0, X_test, y_test)

print(f"folded float test macro-F1   : {F1_FOLD:.4f}")
print(f"W_int8_A8 PTQ test macro-F1  : {f1_ptq:.4f}   (delta {f1_ptq-F1_FOLD:+.4f})")
for n in ["qin", "qa1", "qa2"]:
    L = qm0.get_layer(n)
    print(f"  {n}: bits={L.bits} signed={L.signed} rng={float(L.rng):.4f} scale={float(L.scale()):.6f}")
assert f1_ptq > F1_FOLD - 0.10, "PTQ collapsed; check the wiring"

## 5. Activation range calibration

The activation range decides where clipping happens, and on a heavy-tailed vibration
signal an EMA of the maximum locks onto a rare outlier: at 8 bits that wastes two
thirds of the range, and at 4 bits it collapses the model. The range is therefore
taken from a percentile of the float model's activation distribution.

Two things are separated here, and the order matters:

1. **Clipping alone** is applied to the float model, with bit-widths unchanged
   (`build_clipped`). This isolates what clipping does on its own.
2. **The percentile is selected on the validation split**, never on test, using a
   preservation criterion: minimize the worst deviation of the quantized model from
   the *equally clipped* float model. Maximizing the score instead would run to the
   most aggressive end of the grid, because clipping alone improves cross-load
   accuracy.

The clipping control is what tells us that the accuracy gain belongs to clipping and
not to quantization, which is the substance of the footnote in Table II of the paper.

In [ ]:
# ========================= PERCENTILE RANGE CALIBRATION =========================
import pandas as pd

def _qact_ranges(pctl, n=8192):
    """Activation ranges for qin/qa1/qa2 from a percentile of the float model."""
    idx = np.random.default_rng(SEED).choice(len(X_train), min(n, len(X_train)), replace=False)
    sub = X_train[idx]
    m = models.Model(ref.input, [ref.get_layer("relu1").output, ref.get_layer("relu2").output])
    a1, a2 = m.predict(sub, verbose=0)
    return {"qin": float(np.percentile(np.abs(sub), pctl)),
            "qa1": float(np.percentile(a1, pctl)),
            "qa2": float(np.percentile(a2, pctl))}

def prepare_with(cfg, policy):
    """Config -> quantized model with folded init and the given range policy.
    policy is a percentile, or "max" for the EMA-of-maximum behaviour."""
    qm = load_folded_into(build_quant(cfg))
    if policy == "max":
        calibrate(qm)
    else:
        for n, v in _qact_ranges(pctl=policy).items():
            L = qm.get_layer(n); L.rng.assign(v); L.inited.assign(1.0)
    return qm

### 5.1 Clipping control: is the gain from clipping or from quantization?

In [ ]:
# ========================= CLIPPING CONTROL (float, bit-widths unchanged) =========================
def build_clipped(r_in, r_a1, r_a2):
    inp = layers.Input((WIN, 1), name="sig")
    x = layers.Lambda(lambda t: tf.clip_by_value(t, -r_in, r_in), name="clip_in")(inp)
    x = layers.Conv1D(ARCH["c1_out"], ARCH["c1_k"], strides=ARCH["stride"],
                      padding="same", use_bias=True, name="conv1")(x)
    x = layers.ReLU(max_value=r_a1, name="relu1")(x)
    x = layers.Conv1D(ARCH["c2_out"], ARCH["c2_k"], strides=ARCH["stride"],
                      padding="same", use_bias=True, name="conv2")(x)
    x = layers.ReLU(max_value=r_a2, name="relu2")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    return models.Model(inp, layers.Dense(NUM_CLASSES, name="dense")(x))

def clipped_at(pctl):
    r = _qact_ranges(pctl=pctl)
    m = build_clipped(r["qin"], r["qa1"], r["qa2"])
    src = build_folded(); src.load_weights(FOLDED_PATH)
    for ln in ["conv1", "conv2", "dense"]:
        m.get_layer(ln).set_weights(src.get_layer(ln).get_weights())
    return m, r

print(f"unclipped float : val {macro_f1(ref, X_val, y_val):.4f}  test {F1_FOLD:.4f}\n")
print(f"{'pctl':>7s} {'r_in':>7s} {'r_a1':>7s} {'r_a2':>7s} {'val_F1':>8s} {'test_F1':>8s}")
CLIP_CTRL = {}
for p in [95.0, 97.0, 98.0, 99.0, 99.5, 99.9]:
    m, r = clipped_at(p)
    v, t = macro_f1(m, X_val, y_val), macro_f1(m, X_test, y_test)
    CLIP_CTRL[p] = dict(val=v, test=t, **r)
    print(f"{p:7.2f} {r['qin']:7.3f} {r['qa1']:7.3f} {r['qa2']:7.3f} {v:8.4f} {t:8.4f}")

### 5.2 Selecting the percentile on validation

The grid runs from 95 to 99.99. The selected value must not sit at either end; the
cell warns if it does. Once selected, `RANGE_POLICY` is locked and `prepare` is
bound to it, so every configuration below uses exactly the same calibration.

`F1_BASE_TEST` becomes the accuracy baseline for the study: the float model with the
same clipping applied.

In [ ]:
# ========================= RANGE POLICY SELECTION (preservation criterion, validation) =========================
PCTL_GRID   = [95.0, 97.0, 98.0, 99.0, 99.5, 99.9, 99.99]
ABITS_PROBE = [8, 4, 2]

rows = []
for p in PCTL_GRID:
    ref_clip = clipped_at(p)[0]
    base_val = macro_f1(ref_clip, X_val, y_val)          # equally clipped float baseline
    rec = dict(policy=p, base_val=base_val)
    for a in ABITS_PROBE:
        qm = prepare_with(mk(f"probe_int8_A{a}", "int8", a), p)
        rec[f"A{a}"] = q_macro_f1(qm, X_val, y_val) - base_val   # signed deviation
    rec["worst_abs"] = max(abs(rec[f"A{a}"]) for a in ABITS_PROBE)
    rows.append(rec)

sel = pd.DataFrame(rows).set_index("policy")
print("VALIDATION: quantized minus equally clipped float (closer to 0 is better)\n")
print(sel.round(4).to_string())

BEST = sel["worst_abs"].idxmin()
print(f"\nSELECTED POLICY (preservation, validation): p{BEST}")
if BEST in (PCTL_GRID[0], PCTL_GRID[-1]):
    print("WARNING: selection sits at the edge of the grid; widen it")

RANGE_POLICY = float(BEST)
RANGES = _qact_ranges(pctl=RANGE_POLICY)
def prepare(cfg): return prepare_with(cfg, RANGE_POLICY)

# --- accuracy baseline for the study: float clipped at the same percentile ---
BASE_MODEL = clipped_at(RANGE_POLICY)[0]
F1_BASE_VAL  = macro_f1(BASE_MODEL, X_val,  y_val)
F1_BASE_TEST = macro_f1(BASE_MODEL, X_test, y_test)
print(f"\nlocked policy p{RANGE_POLICY} | ranges {({k: round(v,4) for k,v in RANGES.items()})}")
print(f"ACCURACY BASELINE (clipped float): val {F1_BASE_VAL:.4f}  test {F1_BASE_TEST:.4f}")
print(f"(unclipped float reference       : val {macro_f1(ref, X_val, y_val):.4f}  test {F1_FOLD:.4f})")

## 6. Quantization-aware fine-tuning

First a post-training pass over the whole matrix, with no fine-tuning at all. This
shows which configurations actually need training: with the calibration above, only
the ternary configuration collapses.

Then fine-tuning for a fixed three epochs from the folded initialization, with the
activation ranges frozen, repeated over three seeds. Three epochs is not arbitrary;
the Appendix shows the sweep that fixed it. There is no early stopping: the
validation split is saturated at 0.99 and carries no usable signal.

The final cell locks the accuracy table and writes one weight file per configuration,
using the median seed. Since the weights are not compile-time constants in the
generated RTL, the resource counts do not depend on that choice.

### 6.1 Post-training pass over the matrix

In [ ]:
# ========================= POST-TRAINING PASS OVER THE MATRIX =========================
ptq = []
for name, cfg in EXPERIMENTS.items():
    qm = prepare(cfg)
    ptq.append(dict(
        config = name,
        axis   = cfg["axis"],
        conv2  = cfg["wbits"]["conv2"],
        abit   = cfg["abits"]["qa1"],
        wbits  = weight_bits(cfg["wbits"]),
        val_F1 = q_macro_f1(qm, X_val,  y_val),
        test_F1= q_macro_f1(qm, X_test, y_test)))

P = pd.DataFrame(ptq)
P["d_base"] = P["test_F1"] - F1_BASE_TEST
P = P.sort_values(["axis", "config"])
print(f"calibration p{RANGE_POLICY} | baseline (clipped float) test = {F1_BASE_TEST:.4f}\n")
print(P.round(4).to_string(index=False))

need = P[P["d_base"] < -0.02]["config"].tolist()
print(f"\nneeds fine-tuning (delta < -0.02): {need if need else 'none'}")

### 6.2 Fine-tuning recipe

In [ ]:
# ========================= QUANTIZATION-AWARE FINE-TUNING =========================
QAT_EPOCHS, QAT_LR = 3, 5e-5      # three epochs: see the Appendix sweep
SEEDS = [42, 1, 7]

def qat_finetune(cfg, seed=SEED, epochs=QAT_EPOCHS, verbose=0):
    """Fine-tune from the folded initialization with the activation ranges frozen."""
    tf.keras.utils.set_random_seed(seed)
    qm = prepare(cfg)
    for n in ["qin", "qa1", "qa2"]:
        qm.get_layer(n).trainable = False          # calibration is not a free variable
    qm.compile(optimizer=tf.keras.optimizers.Adam(QAT_LR),
               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    qm.fit(X_train, y_train, epochs=epochs, batch_size=128,
           class_weight=CLASS_WEIGHT, verbose=verbose)
    return qm

### 6.3 Locked accuracy table and deployed weights

In [ ]:
# ========================= LOCK THE ACCURACY TABLE, SAVE DEPLOYED WEIGHTS =========================
final, DEPLOY = [], {}

for name, cfg in EXPERIMENTS.items():
    ptq_f1 = q_macro_f1(prepare(cfg), X_test, y_test)
    te = [q_macro_f1(qat_finetune(cfg, seed=sd), X_test, y_test) for sd in SEEDS]
    med_i  = int(np.argsort(te)[len(te)//2])                 # median seed is deployed
    qm_dep = qat_finetune(cfg, seed=SEEDS[med_i])
    p = os.path.join(DIRS["model"], f"{name}.weights.h5")
    qm_dep.save_weights(p); DEPLOY[name] = p
    final.append(dict(config=name, conv2=cfg["wbits"]["conv2"], abit=cfg["abits"]["qa1"],
                      wbits=weight_bits(cfg["wbits"]), PTQ=ptq_f1,
                      QAT=np.mean(te), std=np.std(te, ddof=1),
                      deploy_seed=SEEDS[med_i],
                      deploy_F1=q_macro_f1(qm_dep, X_test, y_test)))
    print(f"{name:14s} QAT {np.mean(te):.4f} +- {np.std(te, ddof=1):.4f} "
          f"| deployed seed {SEEDS[med_i]} -> {p}")

F = pd.DataFrame(final)
F["d_clipped"]   = F["QAT"] - F1_BASE_TEST
F["d_unclipped"] = F["QAT"] - F1_FOLD
print(f"\nbaselines: clipped float {F1_BASE_TEST:.4f} | unclipped float {F1_FOLD:.4f}\n")
print(F.round(4).to_string(index=False))
print(f"\nband against the unclipped float baseline: {F['d_unclipped'].abs().max():.4f}")
F.to_csv(os.path.join(DIRS["output"], "accuracy_master.csv"), index=False)

## 7. Pure-integer golden reference

The reference reproduces the trained quantized model with integers only, and it is
what the hardware is checked against at every later stage.

Each convolution accumulates integer products and is requantized by a per-channel
multiply-shift fused with ReLU. The shift of each stage is chosen so that the largest
multiplier occupies about 15 bits; a fixed shift loses significance when the
multiplier is small and breaks exactness.

Accumulator widths are fixed analytically from worst-case operand ranges rather than
left to the tool. These widths become `ap_int` widths in the HLS kernel, so they are
part of what the study measures: narrowing the activations narrows the datapath.

The verification target is **exact** agreement of the arg-max between the integer
reference and the quantized Keras model. Anything less and the HLS stage is not worth
starting.

### 7.1 Integer parameter extraction

In [ ]:
# ========================= INTEGER PARAMETER EXTRACTION =========================
def _w_int(w, mode):
    """Bit-exact with quant_weight: returns (integer weights, per-channel scale)."""
    ra = tuple(range(w.ndim - 1)); absw = np.abs(w)
    if mode == "ternary":
        delta = 0.7*absw.mean(axis=ra, keepdims=True)
        mask  = (absw > delta).astype(np.float32)
        alpha = (absw*mask).sum(axis=ra, keepdims=True)/(mask.sum(axis=ra, keepdims=True)+1e-8)
        return (np.sign(w)*mask).astype(np.int64), alpha.ravel().astype(np.float64)
    n = 2**(WBITS_N[mode]-1) - 1
    s = absw.max(axis=ra, keepdims=True)/n + 1e-12
    return np.clip(np.round(w/s), -n, n).astype(np.int64), s.ravel().astype(np.float64)

def pick_shift(M, target_bits=15):
    """Shift such that max(m0) lands near 2^target_bits."""
    return int(np.ceil(target_bits - np.log2(np.max(M))))

def extract_int_params(qm, cfg):
    wb, ab = cfg["wbits"], cfg["abits"]
    a1b, a2b = ab["qa1"], ab["qa2"]
    s_in = float(qm.get_layer("qin").scale())
    s_a1 = float(qm.get_layer("qa1").scale())
    s_a2 = float(qm.get_layer("qa2").scale())

    P = dict(a1_bits=a1b, a2_bits=a2b, s_in=s_in, s_a1=s_a1, s_a2=s_a2,
             n_in=2**7-1, L2=None)
    for ln, s_i, s_o in [("conv1", s_in, s_a1), ("conv2", s_a1, s_a2)]:
        W, b = qm.get_layer(ln).get_weights()
        wi, sc = _w_int(W, wb[ln])
        M  = s_i*sc/s_o
        sh = pick_shift(M)
        P[ln] = dict(w=wi, mode=wb[ln], scale=sc, shift=sh,
                     m0=np.round(M*(2.0**sh)).astype(np.int64),
                     bhi=np.round(b.astype(np.float64)/s_o*(2.0**sh)).astype(np.int64))
    Wd, bd = qm.get_layer("dense").get_weights()
    wdi, sdc = _w_int(Wd, wb["dense"])
    L2 = (WIN // ARCH["stride"]) // ARCH["stride"]
    C  = s_a2*sdc/L2
    Cmax = C.max(); sh = pick_shift(C/Cmax)
    P["L2"] = L2
    P["dense"] = dict(w=wdi, mode=wb["dense"], scale=sdc, shift=sh, Cmax=Cmax,
                      m0 =np.round(C/Cmax*(2.0**sh)).astype(np.int64),
                      bhi=np.round(bd.astype(np.float64)/Cmax*(2.0**sh)).astype(np.int64))
    return P

PARAMS = {}
print(f"{'config':14s} {'layer':7s} {'shift':>5s} {'m0 range':>18s} {'|w|max':>7s}")
for name, cfg in EXPERIMENTS.items():
    qm = build_quant(cfg); qm(X_train[:2]); qm.load_weights(DEPLOY[name])
    P = extract_int_params(qm, cfg); PARAMS[name] = P
    for ln in ["conv1", "conv2", "dense"]:
        d = P[ln]
        print(f"{name if ln=='conv1' else '':14s} {ln:7s} {d['shift']:5d} "
              f"{str((int(d['m0'].min()), int(d['m0'].max()))):>18s} "
              f"{int(np.abs(d['w']).max()):7d}")

### 7.2 Integer forward pass and bit-exact verification

In [ ]:
# ========================= INTEGER FORWARD PASS AND VERIFICATION =========================
def _iconv(x, d, out_bits, stride=2):
    N, Lin, _ = x.shape; k, _, Cout = d["w"].shape
    Lout = -(-Lin // stride); pt = max((Lout-1)*stride + k - Lin, 0); pl = pt//2
    xp  = np.pad(x, ((0,0),(pl, pt-pl),(0,0)))
    idx = np.arange(k)[None,:] + stride*np.arange(Lout)[:,None]
    A   = np.einsum('nlkc,kco->nlo', xp[:, idx, :], d["w"])
    sh  = d["shift"]
    y   = (A*d["m0"][None,None,:] + d["bhi"][None,None,:] + (1 << (sh-1))) >> sh
    return np.clip(y, 0, (1 << out_bits) - 1)

def int_forward(P, x, return_logits=False):
    xi = np.clip(np.round(x.astype(np.float64)/P["s_in"]), -P["n_in"], P["n_in"]).astype(np.int64)
    a1 = _iconv(xi, P["conv1"], P["a1_bits"])
    a2 = _iconv(a1, P["conv2"], P["a2_bits"])
    g  = a2.sum(axis=1)                                   # (N, C2)
    d  = P["dense"]; sh = d["shift"]
    a  = g @ d["w"]                                       # (N, K)
    l  = (a*d["m0"][None,:] + d["bhi"][None,:] + (1 << (sh-1))) >> sh
    return (l.argmax(1), l) if return_logits else l.argmax(1)

def acc_widths(P, cfg):
    n_in, a1n = P["n_in"], (1 << P["a1_bits"]) - 1
    w1 = int(np.abs(P["conv1"]["w"]).max()); w2 = int(np.abs(P["conv2"]["w"]).max())
    wd = int(np.abs(P["dense"]["w"]).max()); a2n = (1 << P["a2_bits"]) - 1
    bits = lambda v: int(np.ceil(np.log2(max(v,1) + 1))) + 1
    A1 = ARCH["c1_k"]*1*n_in*w1
    A2 = ARCH["c2_k"]*ARCH["c1_out"]*a1n*w2
    G  = P["L2"]*a2n
    AD = ARCH["c2_out"]*G*wd
    return dict(conv1_acc=bits(A1), conv2_acc=bits(A2), gap_acc=bits(G),
                dense_acc=bits(AD), dense_mul=bits(AD)+bits(int(P["dense"]["m0"].max())))

print(f"{'config':14s} {'golden F1':>9s} {'QAT F1':>7s} {'argmax match':>13s} "
      f"{'c1':>4s} {'c2':>4s} {'gap':>4s} {'dns':>4s} {'d*m0':>5s}")
WIDTHS = {}
for name, cfg in EXPERIMENTS.items():
    P  = PARAMS[name]
    qm = build_quant(cfg); qm(X_train[:2]); qm.load_weights(DEPLOY[name])
    pq = qm.predict(X_test, verbose=0).argmax(1)
    pg = int_forward(P, X_test)
    w  = acc_widths(P, cfg); WIDTHS[name] = w
    print(f"{name:14s} {f1_score(y_test,pg,average='macro'):9.4f} "
          f"{f1_score(y_test,pq,average='macro'):7.4f} "
          f"{100*(pg==pq).mean():12.2f}% "
          f"{w['conv1_acc']:4d} {w['conv2_acc']:4d} {w['gap_acc']:4d} "
          f"{w['dense_acc']:4d} {w['dense_mul']:5d}")

## 8. HLS artifacts

Everything the hardware flow needs is emitted from here, one folder per configuration
under `hls/`.

- `weights.h` carries the integer weights, the requantization constants, the
  analytically fixed accumulator widths and the per-layer mode flags.
- `test_data.h` carries class-balanced test vectors with the golden class and golden
  logits, so C simulation can check itself.
- `cnn.h` / `cnn.cpp` are a single source serving every precision; a compile-time flag
  selects the multiplier-free sign path or the integer-multiply path.
- All directives live in `run.tcl`, never as pragmas in the source, so the same
  directive set serves every configuration and any resource difference is
  attributable to precision alone.
- `cnn_hw.cpp` wraps the kernel with an AXI master data port and an AXI-Lite control
  port for board deployment.

Two implementation details are deliberate. Loop nests are flattened by hand, and all
index arithmetic is kept in narrow `ap_int` types; leaving either to the tool
introduces wide multipliers and remainder operations into the address path, which in
an earlier version of this design consumed 16 DSP slices on address computation alone.

### 8.1 `weights.h` and `test_data.h`

In [ ]:
# ========================= EMIT weights.h AND test_data.h =========================
MODE_CODE = {"int8": 8, "int4": 4, "int2": 1, "ternary": 1}   # 1 = multiplier-free {-1,0,+1}
N_PER_CLASS = 10                                              # test vectors per class

def _ctype(a):
    m = int(np.abs(np.asarray(a)).max())
    for t, lim in [("int8_t",127), ("int16_t",32767), ("int32_t",2**31-1)]:
        if m <= lim: return t
    return "int64_t"

def c_arr(name, arr, ctype=None):
    a = np.asarray(arr); ctype = ctype or _ctype(a)
    dims = "".join(f"[{d}]" for d in a.shape)
    def fmt(x):
        if x.ndim == 1: return "{" + ",".join(str(int(v)) for v in x) + "}"
        return "{" + ",\n".join(fmt(s) for s in x) + "}"
    return f"static const {ctype} {name}{dims} = {fmt(a)};\n"

def emit_weights_h(name, cfg, P, W, path):
    c1, c2, dn = P["conv1"], P["conv2"], P["dense"]
    s = ["#ifndef WEIGHTS_H", "#define WEIGHTS_H", "#include <stdint.h>", ""]
    s += [f"// config: {name}  w={cfg['wbits']}  a1/a2={P['a1_bits']}/{P['a2_bits']}", ""]
    s += [f"#define IN_LEN {WIN}", f"#define C1_COUT {ARCH['c1_out']}",
          f"#define C1_K {ARCH['c1_k']}", f"#define C2_COUT {ARCH['c2_out']}",
          f"#define C2_K {ARCH['c2_k']}", f"#define N_CLASS {NUM_CLASSES}",
          f"#define STRIDE {ARCH['stride']}", ""]
    s += [f"#define A1_BITS {P['a1_bits']}", f"#define A2_BITS {P['a2_bits']}", ""]
    s += [f"#define MODE_C1 {MODE_CODE[cfg['wbits']['conv1']]}",
          f"#define MODE_C2 {MODE_CODE[cfg['wbits']['conv2']]}",
          f"#define MODE_D  {MODE_CODE[cfg['wbits']['dense']]}", ""]
    s += [f"#define ACC1_W {W['conv1_acc']}", f"#define ACC2_W {W['conv2_acc']}",
          f"#define GAP_W  {W['gap_acc']}",   f"#define DACC_W {W['dense_acc']}",
          f"#define DMUL_W {W['dense_mul']}", ""]
    s += [f"#define SH1 {c1['shift']}", f"#define SH2 {c2['shift']}",
          f"#define SHD {dn['shift']}", ""]
    s += [c_arr("W1", c1["w"]),  c_arr("M01", c1["m0"]),  c_arr("B1", c1["bhi"]),
          c_arr("W2", c2["w"]),  c_arr("M02", c2["m0"]),  c_arr("B2", c2["bhi"]),
          c_arr("WD", dn["w"]),  c_arr("M0D", dn["m0"]),  c_arr("BD", dn["bhi"]),
          "#endif\n"]
    open(path, "w").write("\n".join(s))

def pick_balanced(y, n_per, seed=SEED):
    rng = np.random.default_rng(seed); idx = []
    for c in range(NUM_CLASSES):
        ci = np.where(y == c)[0]
        idx += list(rng.choice(ci, min(n_per, len(ci)), replace=False))
    return np.array(sorted(idx))

def emit_test_data_h(P, path, n_per=N_PER_CLASS):
    sel = pick_balanced(y_test, n_per)
    xq  = np.clip(np.round(X_test[sel].astype(np.float64)/P["s_in"]),
                  -P["n_in"], P["n_in"]).astype(np.int64)[..., 0]
    gc, gl = int_forward(P, X_test[sel], return_logits=True)
    s = ["#ifndef TEST_DATA_H", "#define TEST_DATA_H", "#include <stdint.h>", "",
         f"#define N_TEST {len(sel)}", "",
         c_arr("TEST_X", xq, "int8_t"),
         c_arr("GOLD_CLS", gc.astype(np.int64), "int32_t"),
         c_arr("GOLD_LOGIT", gl, "int64_t"), "#endif\n"]
    open(path, "w").write("\n".join(s))
    return len(sel), gc, y_test[sel]

HLS_ROOT = DIRS["hls"]
for name, cfg in EXPERIMENTS.items():
    d = os.path.join(HLS_ROOT, name); os.makedirs(d, exist_ok=True)
    emit_weights_h(name, cfg, PARAMS[name], WIDTHS[name], os.path.join(d, "weights.h"))
    n, gc, yt = emit_test_data_h(PARAMS[name], os.path.join(d, "test_data.h"))
    kb = sum(os.path.getsize(os.path.join(d, f))//1024 for f in ["weights.h","test_data.h"])
    print(f"{name:14s} -> {d}  ({n} vectors, golden accuracy {(gc==yt).mean():.3f}, {kb} KB)")

### 8.2 HLS kernel, testbench and Tcl directives

In [ ]:
# ========================= EMIT cnn.h / cnn.cpp / tb.cpp / run.tcl =========================
CNN_H = r'''#ifndef CNN_H
#define CNN_H
#include <ap_int.h>
#include "weights.h"

#define L1     ((IN_LEN + STRIDE - 1)/STRIDE)
#define L2_LEN ((L1 + STRIDE - 1)/STRIDE)
#define PAD1   (((L1-1)*STRIDE + C1_K - IN_LEN)/2)
#define PAD2   (((L2_LEN-1)*STRIDE + C2_K - L1)/2)

#ifndef UNROLL_C2
#define UNROLL_C2 1
#endif
#ifndef UNROLL_C1
#define UNROLL_C1 1
#endif

#if (C2_COUT % UNROLL_C2) != 0
#error "UNROLL_C2 must divide C2_COUT"
#endif
#if (C1_COUT % UNROLL_C1) != 0
#error "UNROLL_C1 must divide C1_COUT"
#endifROLL_C2 must divide C2_COUT"
#endif

typedef ap_int<8>           i8_t;
typedef ap_uint<A1_BITS>    a1_t;
typedef ap_uint<A2_BITS>    a2_t;
typedef ap_int<ACC1_W>      acc1_t;
typedef ap_int<ACC2_W>      acc2_t;
typedef ap_int<ACC1_W+20>   r1_t;
typedef ap_int<ACC2_W+20>   r2_t;
typedef ap_uint<GAP_W>      gap_t;
typedef ap_int<DACC_W>      dacc_t;
typedef ap_int<DMUL_W+8>    dmul_t;
typedef ap_uint<12>  o1_t;
typedef ap_uint<11>  o2_t;
typedef ap_int<14>   sidx_t;

void cnn_top(const i8_t in[IN_LEN], int *cls, ap_int<64> logit_out[N_CLASS]);
#endif
'''

CNN_CPP = r'''#include "cnn.h"

static a1_t rq1(acc1_t acc, int co) {
  r1_t t = (r1_t)acc * (r1_t)M01[co] + (r1_t)B1[co] + ((r1_t)1 << (SH1-1));
  r1_t y = t >> SH1;
  const r1_t hi = ((r1_t)1 << A1_BITS) - 1;
  if (y < 0) y = 0; else if (y > hi) y = hi;
  return (a1_t)y;
}

static a2_t rq2(acc2_t acc, int co) {
  r2_t t = (r2_t)acc * (r2_t)M02[co] + (r2_t)B2[co] + ((r2_t)1 << (SH2-1));
  r2_t y = t >> SH2;
  const r2_t hi = ((r2_t)1 << A2_BITS) - 1;
  if (y < 0) y = 0; else if (y > hi) y = hi;
  return (a2_t)y;
}

static void load_in(const i8_t in[IN_LEN], i8_t buf[IN_LEN]) {
  LD: for (int i = 0; i < IN_LEN; i++) buf[i] = in[i];
}

static void conv1(const i8_t buf[IN_LEN], a1_t a1[L1][C1_COUT]) {
  const int NB1 = C1_COUT / UNROLL_C1;
  o1_t o = 0; ap_uint<5> cb = 0;
  C1_MAIN: for (int i = 0; i < L1*NB1; i++) {
    acc1_t acc[UNROLL_C1];
    C1_Z: for (int u = 0; u < UNROLL_C1; u++) acc[u] = 0;
    C1_K_L: for (int k = 0; k < C1_K; k++) {
      sidx_t t = (sidx_t)(o << 1) + (sidx_t)k - (sidx_t)PAD1;
      i8_t x = (t < 0 || t >= (sidx_t)IN_LEN) ? (i8_t)0 : buf[t];
      C1_U: for (int u = 0; u < UNROLL_C1; u++)
        acc[u] += (acc1_t)(x * W1[k][0][cb*UNROLL_C1+u]);
    }
    C1_W: for (int u = 0; u < UNROLL_C1; u++) {
      int co = cb*UNROLL_C1 + u;
      a1[o][co] = rq1(acc[u], co);
    }
    if (cb == NB1-1) { cb = 0; o++; } else { cb++; }
  }
}

static void conv2(const a1_t a1[L1][C1_COUT], a2_t a2[L2_LEN][C2_COUT]) {
  const int NB = C2_COUT / UNROLL_C2;
  o2_t o = 0; ap_uint<6> cb = 0;
  C2_MAIN: for (int i = 0; i < L2_LEN*NB; i++) {
    acc2_t acc[UNROLL_C2];
    C2_Z: for (int u = 0; u < UNROLL_C2; u++) acc[u] = 0;
    C2_K_L: for (int k = 0; k < C2_K; k++) {
      sidx_t t = (sidx_t)(o << 1) + (sidx_t)k - (sidx_t)PAD2;
      bool ok = (t >= 0 && t < (sidx_t)L1);
      o1_t ta = ok ? (o1_t)t : (o1_t)0;
      C2_CIN: for (int ci = 0; ci < C1_COUT; ci++) {
        a1_t x = ok ? a1[ta][ci] : (a1_t)0;
        C2_U: for (int u = 0; u < UNROLL_C2; u++) {
          int co = cb*UNROLL_C2 + u;
#if MODE_C2 == 1
          int8_t w = W2[k][ci][co];
          if      (w > 0) acc[u] += (acc2_t)x;
          else if (w < 0) acc[u] -= (acc2_t)x;
#else
          acc[u] += (acc2_t)(x * W2[k][ci][co]);
#endif
        }
      }
    }
    C2_W: for (int u = 0; u < UNROLL_C2; u++) {
      int co = cb*UNROLL_C2 + u;
      a2[o][co] = rq2(acc[u], co);
    }
    if (cb == NB-1) { cb = 0; o++; } else { cb++; }
  }
}

static void gap(const a2_t a2[L2_LEN][C2_COUT], gap_t g[C2_COUT]) {
  gap_t s[C2_COUT];
  G_Z: for (int c = 0; c < C2_COUT; c++) s[c] = 0;
  G_L: for (int t = 0; t < L2_LEN; t++) {
    G_CH: for (int c = 0; c < C2_COUT; c++) s[c] += a2[t][c];
  }
  G_O: for (int c = 0; c < C2_COUT; c++) g[c] = s[c];
}

static void dense_argmax(const gap_t g[C2_COUT], int *cls, ap_int<64> logit_out[N_CLASS]) {
  ap_int<64> best = -(((ap_int<64>)1) << 62); int bk = 0;
  D_K: for (int k = 0; k < N_CLASS; k++) {
    dacc_t a = 0;
    D_IN: for (int j = 0; j < C2_COUT; j++) {
      a += (dacc_t)((int)g[j] * (int)WD[j][k]);
    }
    dmul_t t = (dmul_t)a * (dmul_t)M0D[k] + (dmul_t)BD[k] + ((dmul_t)1 << (SHD-1));
    ap_int<64> l = (ap_int<64>)(t >> SHD);
    logit_out[k] = l;
    if (l > best) { best = l; bk = k; }
  }
  *cls = bk;
}

void cnn_top(const i8_t in[IN_LEN], int *cls, ap_int<64> logit_out[N_CLASS]) {
  i8_t  buf[IN_LEN];
  a1_t  a1[L1][C1_COUT];
  a2_t  a2[L2_LEN][C2_COUT];
  gap_t g[C2_COUT];
  load_in(in, buf);
  conv1(buf, a1);
  conv2(a1, a2);
  gap(a2, g);
  dense_argmax(g, cls, logit_out);
}
'''

TB_CPP = r'''#include <cstdio>
#include "cnn.h"
#include "test_data.h"
int main() {
  int mc = 0, ml = 0;
  for (int n = 0; n < N_TEST; n++) {
    i8_t in[IN_LEN];
    for (int i = 0; i < IN_LEN; i++) in[i] = (i8_t)TEST_X[n][i];
    int cls; ap_int<64> lo[N_CLASS];
    cnn_top(in, &cls, lo);
    if (cls != GOLD_CLS[n]) mc++;
    for (int k = 0; k < N_CLASS; k++)
      if (lo[k] != (ap_int<64>)GOLD_LOGIT[n][k]) ml++;
  }
  printf("argmax mismatch: %d / %d\n", mc, N_TEST);
  printf("logit  mismatch: %d / %d\n", ml, N_TEST*N_CLASS);
  return (mc || ml) ? 1 : 0;
}
'''

RUN_TCL = r'''# usage:  vitis_hls -f run.tcl -tclargs <UNROLL_C2>
# or:     UNROLL=4 vitis_hls -f run.tcl
set U 1
if {[info exists ::env(UNROLL)]} { set U $::env(UNROLL) }
foreach a $argv { if {[string is integer -strict $a]} { set U $a } }
puts "== UNROLL_C2 = $U =="
set FL "-I. -DUNROLL_C2=$U"

open_project -reset proj_U$U
set_top cnn_top
add_files cnn.cpp -cflags $FL
add_files -tb tb.cpp -cflags $FL
open_solution -reset sol1 -flow_target vivado
set_part {xc7z020clg400-1}
create_clock -period 20 -name default

set_directive_pipeline -II 1 "load_in/LD"
set_directive_array_partition -type cyclic -factor 8 -dim 1 "cnn_top" buf

set_directive_dataflow "cnn_top"

set_directive_pipeline -II 1 "conv1/C1_MAIN"
set_directive_unroll             "conv1/C1_K_L"

set_directive_pipeline -II 1 "conv2/C2_MAIN"
set_directive_unroll             "conv2/C2_K_L"
set_directive_unroll             "conv2/C2_CIN"
set_directive_unroll             "conv2/C2_U"

set_directive_pipeline -II 1 "gap/G_L"
set_directive_unroll             "gap/G_CH"
set_directive_pipeline -II 1 "dense_argmax/D_IN"

set_directive_array_partition -type complete -dim 0 "conv1"       W1
set_directive_array_partition -type complete -dim 1 "conv2"       W2
set_directive_array_partition -type complete -dim 2 "conv2"       W2
set_directive_array_partition -type cyclic   -factor 8 -dim 1 "cnn_top" buf
set_directive_array_partition -type complete -dim 2        "cnn_top" a1
set_directive_array_partition -type complete -dim 2        "cnn_top" a2
set_directive_array_partition -type complete -dim 1 "gap"         s

csim_design
csynth_design
close_project
exit
'''

# --- integrity check on the emitted sources ---
for name in EXPERIMENTS:
    p = os.path.join(HLS_ROOT, name, "cnn.cpp"); s = open(p).read()
    ok = s.count("{") == s.count("}") and s.rstrip().endswith("}")
    print(f"{name:14s} cnn.cpp {len(s.splitlines()):3d} lines  "
          f"{{{s.count('{')}/{s.count('}')}}}  {'OK' if ok else 'BROKEN'}")
    print(f"{name:14s} -> cnn.h cnn.cpp tb.cpp run.tcl")
print("\nrun with:  cd <config folder> && vitis_hls -f run.tcl -tclargs 1")

### 8.3 Batch synthesis and report parsing

In [ ]:
# ========================= EMIT run_all.tcl (batch synthesis) =========================
RUN_ALL = r'''# vitis_hls -f run_all.tcl     (run from the hls/ root)
set ROOT [file normalize [file dirname [info script]]]
set CONFIGS {W_int8_A8 W_int4_A8 W_ternary_A8 A_int8_A4 A_int4_A4}
set UNROLLS {1}
set CSV [file join $ROOT results.csv]

proc num {s} {
  if {[regexp {(-?\d+)} $s -> v]} { return $v }
  return ""
}
proc row {txt name} {
  foreach ln [split $txt "\n"] {
    set ln [string trim $ln]
    if {![string match "|*" $ln]} continue
    set f [split $ln "|"]
    if {[llength $f] < 14} continue
    set nm [string trim [string map {+ "" o "" * ""} [lindex $f 1]]]
    if {$nm eq $name} { return $f }
  }
  return {}
}

set fh [open $CSV w]
puts $fh "config,unroll,csim_argmax,csim_logit,latency,II_top,DSP,LUT,FF,BRAM,conv2_DSP,conv2_LUT,C2_II,conv1_DSP,C1_II"
foreach cfg $CONFIGS {
  foreach U $UNROLLS {
    set d [file join $ROOT $cfg]
    if {![file isdirectory $d]} { puts "ATLA: $d yok"; continue }
    cd $d
    puts "=== $cfg  U=$U ==="
    set FL "-I. -DUNROLL_C2=$U"
    set p proj_U$U
    open_project -reset $p
    set_top cnn_top
    add_files cnn.cpp -cflags $FL
    add_files -tb tb.cpp -cflags $FL
    open_solution -reset sol1 -flow_target vivado
    set_part {xc7z020clg400-1}
    create_clock -period 20 -name default

    set_directive_dataflow "cnn_top"
    set_directive_pipeline -II 1 "load_in/LD"
    set_directive_pipeline -II 1 "conv1/C1_MAIN"
    set_directive_unroll             "conv1/C1_K_L"
    set_directive_pipeline -II 1 "conv2/C2_MAIN"
    set_directive_unroll             "conv2/C2_K_L"
    set_directive_unroll             "conv2/C2_CIN"
    set_directive_unroll             "conv2/C2_U"
    set_directive_pipeline -II 1 "gap/G_L"
    set_directive_pipeline -II 1 "dense_argmax/D_IN"
    set_directive_array_partition -type complete -dim 0 "conv1" W1
    set_directive_array_partition -type complete -dim 1 "conv2" W2
    set_directive_array_partition -type complete -dim 2 "conv2" W2
    set_directive_array_partition -type cyclic -factor 8 -dim 1 "cnn_top" buf
    set_directive_array_partition -type complete -dim 2 "cnn_top" a1
    set_directive_array_partition -type complete -dim 2 "cnn_top" a2
    set_directive_array_partition -type complete -dim 1 "gap" s

    csim_design
    csynth_design
    close_project

    # --- parse ---
    set ma ""; set ml ""
    set lg [file join $d $p sol1 csim report cnn_top_csim.log]
    if {[file exists $lg]} {
      set t [read [set f [open $lg]]]; close $f
      regexp {argmax mismatch:\s*(\d+)} $t -> ma
      regexp {logit\s+mismatch:\s*(\d+)} $t -> ml
    }
    set rp [file join $d $p sol1 syn report csynth.rpt]
    set lat ""; set ii ""; set dsp ""; set lut ""; set ff ""; set bram ""
    set c2d ""; set c2l ""; set c2ii ""; set c1d ""; set c1ii ""
    if {[file exists $rp]} {
      set t [read [set f [open $rp]]]; close $f
      set r [row $t "cnn_top"]
      if {[llength $r]} {
        set lat  [num [lindex $r 4]];  set ii   [num [lindex $r 7]]
        set bram [num [lindex $r 10]]; set dsp  [num [lindex $r 11]]
        set ff   [num [lindex $r 12]]; set lut  [num [lindex $r 13]]
      }
      set r [row $t "conv2"]
      if {[llength $r]} { set c2d [num [lindex $r 11]]; set c2l [num [lindex $r 13]] }
      set r [row $t "C2_MAIN"];  if {[llength $r]} { set c2ii [num [lindex $r 7]] }
      set r [row $t "conv1"];    if {[llength $r]} { set c1d  [num [lindex $r 11]] }
      set r [row $t "C1_MAIN"];  if {[llength $r]} { set c1ii [num [lindex $r 7]] }
    }
    puts $fh "$cfg,$U,$ma,$ml,$lat,$ii,$dsp,$lut,$ff,$bram,$c2d,$c2l,$c2ii,$c1d,$c1ii"
    flush $fh
    cd $ROOT
  }
}
close $fh
puts "\n>>> yazildi: $CSV"
exit
'''
open(os.path.join(HLS_ROOT, "run_all.tcl"), "w").write(RUN_ALL)

# --- copy run.tcl into every config folder (one directive set for all) ---
src = open(os.path.join(HLS_ROOT, "W_int8_A8", "run.tcl")).read()
for name in EXPERIMENTS:
    open(os.path.join(HLS_ROOT, name, "run.tcl"), "w").write(src)

print(f"run_all.tcl -> {os.path.join(HLS_ROOT, 'run_all.tcl')}")
print("run with:    cd hls  &&  vitis_hls -f run_all.tcl")
print("output:      hls/results.csv")

In [ ]:
# ========================= EMIT parse_all.tcl (parse only, no synthesis) =========================
PARSE_ALL = r'''# vitis_hls -f parse_all.tcl     (from the hls/ root; does NOT run synthesis)
set ROOT [file normalize [file dirname [info script]]]
set CONFIGS {W_int8_A8 W_int4_A8 W_ternary_A8 A_int8_A4 A_int4_A4}
set UNROLLS {1}

proc num {s} { if {[regexp {(-?\d+)} $s -> v]} { return $v } ; return "" }

proc row {txt name} {
  foreach ln [split $txt "\n"] {
    set ln [string trim $ln]
    if {![string match "|*" $ln]} continue
    set f [split $ln "|"]
    if {[llength $f] < 14} continue
    set nm [string trim [lindex $f 1]]
    regsub {^[+o*]\s*} $nm "" nm
    set nm [string trimright [string trim $nm] "*"]
    if {$nm eq $name} { return $f }
  }
  return {}
}

set fh [open [file join $ROOT results.csv] w]
puts $fh "config,unroll,csim_argmax,csim_logit,latency,II_top,DSP,LUT,FF,BRAM,conv2_DSP,conv2_LUT,C2_II,conv1_DSP,C1_II"
foreach cfg $CONFIGS {
  foreach U $UNROLLS {
    set d [file join $ROOT $cfg]
    set p proj_U$U
    set ma ""; set ml ""
    foreach cand [list [file join $d $p sol1 csim report cnn_top_csim.log] \
                       [file join $d $p sol1 csim build csim.log]] {
      if {[file exists $cand]} {
        set t [read [set f [open $cand]]]; close $f
        regexp {argmax mismatch:\s*(\d+)} $t -> ma
        regexp {logit\s+mismatch:\s*(\d+)} $t -> ml
        break
      }
    }
    set rp [file join $d $p sol1 syn report csynth.rpt]
    foreach v {lat ii dsp lut ff bram c2d c2l c2ii c1d c1ii} { set $v "" }
    if {[file exists $rp]} {
      set t [read [set f [open $rp]]]; close $f
      set r [row $t "cnn_top"]
      if {[llength $r]} {
        set lat [num [lindex $r 4]];  set ii   [num [lindex $r 7]]
        set bram [num [lindex $r 10]]; set dsp [num [lindex $r 11]]
        set ff  [num [lindex $r 12]]; set lut  [num [lindex $r 13]]
      }
      set r [row $t "conv2"]
      if {[llength $r]} { set c2d [num [lindex $r 11]]; set c2l [num [lindex $r 13]] }
      set r [row $t "conv1"]   ; if {[llength $r]} { set c1d  [num [lindex $r 11]] }
      set r [row $t "C2_MAIN"] ; if {[llength $r]} { set c2ii [num [lindex $r 7]] }
      set r [row $t "C1_MAIN"] ; if {[llength $r]} { set c1ii [num [lindex $r 7]] }
    } else { puts "YOK: $rp" }
    puts $fh "$cfg,$U,$ma,$ml,$lat,$ii,$dsp,$lut,$ff,$bram,$c2d,$c2l,$c2ii,$c1d,$c1ii"
  }
}
close $fh
puts ">>> yazildi: [file join $ROOT results.csv]"
exit
'''
open(os.path.join(HLS_ROOT, "parse_all.tcl"), "w").write(PARSE_ALL)

# --- patch the module-name parser in run_all.tcl as well ---
p = os.path.join(HLS_ROOT, "run_all.tcl"); s = open(p).read()
s = s.replace(
  '    set nm [string trim [string map {+ "" o "" * ""} [lindex $f 1]]]\n',
  '    set nm [string trim [lindex $f 1]]\n'
  '    regsub {^[+o*]\\s*} $nm "" nm\n'
  '    set nm [string trimright [string trim $nm] "*"]\n')
open(p, "w").write(s)
print("parse_all.tcl written")

### 8.4 AXI wrapper and IP export

In [ ]:
# ========================= EMIT cnn_hw.cpp / tb_hw.cpp / hw.tcl (AXI wrapper) =========================
CNN_HW_CPP = r'''#include "cnn.h"
#include <string.h>

// cnn.cpp icindeki asamalar
void cnn_core(const i8_t in[IN_LEN], int *cls, ap_int<64> logit_out[N_CLASS]);

extern "C" void cnn_hw(const int8_t *x_in, int32_t *y_out, int n_win) {
#pragma HLS INTERFACE m_axi     port=x_in  offset=slave bundle=gmem depth=2048
#pragma HLS INTERFACE m_axi     port=y_out offset=slave bundle=gmem depth=11
#pragma HLS INTERFACE s_axilite port=x_in  bundle=control
#pragma HLS INTERFACE s_axilite port=y_out bundle=control
#pragma HLS INTERFACE s_axilite port=n_win bundle=control
#pragma HLS INTERFACE s_axilite port=return bundle=control

  WIN_LOOP: for (int w = 0; w < n_win; w++) {
    i8_t local[IN_LEN];
    CP_IN: for (int i = 0; i < IN_LEN; i++) {
#pragma HLS PIPELINE II=1
      local[i] = (i8_t)x_in[(long)w*IN_LEN + i];
    }
    int cls; ap_int<64> lo[N_CLASS];
    cnn_core(local, &cls, lo);
    y_out[(long)w*(N_CLASS+1)] = (int32_t)cls;
    CP_OUT: for (int k = 0; k < N_CLASS; k++) {
#pragma HLS PIPELINE II=1
      y_out[(long)w*(N_CLASS+1) + 1 + k] = (int32_t)lo[k];
    }
  }
}
'''

TB_HW_CPP = r'''#include <cstdio>
#include <cstdlib>
#include "cnn.h"
#include "test_data.h"
extern "C" void cnn_hw(const int8_t*, int32_t*, int);
int main() {
  int8_t  *x = (int8_t*) malloc((size_t)N_TEST*IN_LEN);
  int32_t *y = (int32_t*)malloc((size_t)N_TEST*(N_CLASS+1)*sizeof(int32_t));
  for (int n = 0; n < N_TEST; n++)
    for (int i = 0; i < IN_LEN; i++) x[(size_t)n*IN_LEN+i] = (int8_t)TEST_X[n][i];
  cnn_hw(x, y, N_TEST);
  int mc = 0, ml = 0;
  for (int n = 0; n < N_TEST; n++) {
    if (y[(size_t)n*(N_CLASS+1)] != GOLD_CLS[n]) mc++;
    for (int k = 0; k < N_CLASS; k++)
      if ((int64_t)y[(size_t)n*(N_CLASS+1)+1+k] != GOLD_LOGIT[n][k]) ml++;
  }
  printf("HW argmax mismatch: %d / %d\n", mc, N_TEST);
  printf("HW logit  mismatch: %d / %d\n", ml, N_TEST*N_CLASS);
  free(x); free(y);
  return (mc || ml) ? 1 : 0;
}
'''

HW_TCL = r'''# vitis_hls -f hw.tcl -tclargs 1      (from a config folder; exports the IP)
set U 1
foreach a $argv { if {[string is integer -strict $a]} { set U $a } }
set FL "-I. -DUNROLL_C2=$U -DUNROLL_C1=1"

open_project -reset hw_U$U
set_top cnn_hw
add_files cnn.cpp   -cflags $FL
add_files cnn_hw.cpp -cflags $FL
add_files -tb tb_hw.cpp -cflags $FL
open_solution -reset sol1 -flow_target vivado
set_part {xc7z020clg400-1}
create_clock -period 20 -name default

set_directive_dataflow "cnn_core"
set_directive_pipeline -II 1 "load_in/LD"
set_directive_pipeline -II 1 "conv1/C1_MAIN"
set_directive_unroll             "conv1/C1_K_L"
set_directive_unroll             "conv1/C1_U"
set_directive_pipeline -II 1 "conv2/C2_MAIN"
set_directive_unroll             "conv2/C2_K_L"
set_directive_unroll             "conv2/C2_CIN"
set_directive_unroll             "conv2/C2_U"
set_directive_pipeline -II 1 "gap/G_L"
set_directive_pipeline -II 1 "dense_argmax/D_IN"
set_directive_array_partition -type complete -dim 0 "conv1" W1
set_directive_array_partition -type complete -dim 1 "conv2" W2
set_directive_array_partition -type complete -dim 2 "conv2" W2
set_directive_array_partition -type cyclic -factor 8 -dim 1 "cnn_core" buf
set_directive_array_partition -type complete -dim 2 "cnn_core" a1
set_directive_array_partition -type complete -dim 2 "cnn_core" a2
set_directive_array_partition -type complete -dim 1 "gap" s

csim_design
csynth_design
export_design -format ip_catalog -rtl verilog
close_project
exit
'''

# rename cnn_top -> cnn_core so the wrapper can call it
for name in EXPERIMENTS:
    d = os.path.join(HLS_ROOT, name)
    src = open(os.path.join(d, "cnn.cpp")).read()
    src = src.replace("void cnn_top(", "void cnn_core(")
    open(os.path.join(d, "cnn.cpp"), "w").write(src)
    h = open(os.path.join(d, "cnn.h")).read()
    h = h.replace("void cnn_top(", "void cnn_core(")
    open(os.path.join(d, "cnn.h"), "w").write(h)
    for fn, txt in [("cnn_hw.cpp", CNN_HW_CPP), ("tb_hw.cpp", TB_HW_CPP), ("hw.tcl", HW_TCL)]:
        open(os.path.join(d, fn), "w").write(txt)
    print(f"{name:14s} -> cnn_hw.cpp tb_hw.cpp hw.tcl  (cnn_top -> cnn_core)")

print("\nNote: tb.cpp also calls cnn_top; updating it as well")
for name in EXPERIMENTS:
    p = os.path.join(HLS_ROOT, name, "tb.cpp")
    open(p, "w").write(open(p).read().replace("cnn_top(", "cnn_core("))
print("done")

## 9. Board export

One `.npz` per configuration holding the int8-quantized test windows, the golden
class decisions and the golden logits. Copy these to the PYNQ-Z1 alongside
`<config>.bit` and `<config>.hwh`, then run `02_pynq_measure.ipynb`.

In [ ]:
# ========================= EXPORT GOLDEN VECTORS FOR THE BOARD =========================
for name, cfg in EXPERIMENTS.items():
    P  = PARAMS[name]
    xq = np.clip(np.round(X_test.astype(np.float64)/P["s_in"]),
                 -P["n_in"], P["n_in"]).astype(np.int8)[..., 0]
    gc, gl = int_forward(P, X_test, return_logits=True)
    p = os.path.join(DIRS["output"], f"golden_{name}.npz")
    np.savez_compressed(p, x=xq, cls=gc.astype(np.int32),
                        logit=gl.astype(np.int64), y=y_test.astype(np.int32),
                        latency_cycles=86959, clk_mhz=50)
    print(f"{name:14s} -> {p}  ({xq.shape[0]} pencere, "
          f"golden accuracy {(gc==y_test).mean():.4f}, "
          f"{os.path.getsize(p)//1024} KB)")

## 10. Figures

Two of the manuscript figures are produced here. Figure 1(a), the flow diagram, is
drawn in draw.io and lives in the repository as `figures/fig_flow.drawio`.

The DSP figures below are the measured results, not recomputed here: HLS estimates
come from the `csynth.rpt` files and the built counts from the Vivado netlists, both
collected by the Tcl scripts of Section 8 and stored under `results/`.

### 10.1 HLS estimate against the synthesized netlist

In [ ]:
# ========================= FIGURE: HLS ESTIMATE VS SYNTHESIZED NETLIST =========================
import matplotlib.pyplot as plt
import numpy as np

cfg  = ["W8A8", "W4A8", "W8A4", "W4A4", "TerA8"]
hls  = [59, 55, 59, 54, 9]      # from results/csynth/*_csynth.rpt
impl = [89,  9, 88,  8, 9]      # from results/harvest.csv

x = np.arange(len(cfg)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 3.5), dpi=300)

b1 = ax.bar(x - w/2, hls,  w, label="HLS Estimate",
            color="#4C72B0", alpha=0.9, edgecolor="none", zorder=3)
b2 = ax.bar(x + w/2, impl, w, label="Built (Synthesis)",
            color="#DD8452", alpha=0.9, edgecolor="none", zorder=3)

for b in list(b1) + list(b2):
    h = b.get_height()
    ax.annotate(f"{int(h)}", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=9, fontweight="semibold", color="#333333")

ax.set_xticks(x); ax.set_xticklabels(cfg, fontsize=10, fontweight="medium")
ax.set_ylabel("DSP48 Slices", fontsize=11, fontweight="medium")
ax.set_ylim(0, 105)
ax.tick_params(axis="y", labelsize=9)
ax.legend(fontsize=10, frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.spines["left"].set_color("#cccccc"); ax.spines["bottom"].set_color("#cccccc")
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()

out = os.path.join(DIRS["fig"], "fig_dsp_gap.png")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("written:", out)

### 10.2 Measurement setup photograph

Annotates the raw photograph `fig/set.png` with the connection labels used in
Figure 1(b) of the manuscript. The unannotated photograph is in the repository so the
labels can be checked against it.

In [ ]:
# ========================= FIGURE: ANNOTATED MEASUREMENT SETUP =========================
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

SRC_IMG = os.path.join(DIRS["fig"], "set.png")          # raw photograph
OUT_IMG = os.path.join(DIRS["fig"], "fig_setup.png")

im = Image.open(SRC_IMG).convert("RGB").crop((0, 10, 1988, 900))
fig, ax = plt.subplots(figsize=(7.0, 7.0 * im.size[1] / im.size[0]))
ax.imshow(np.asarray(im)); ax.axis("off")

BB = dict(boxstyle="round,pad=0.20", fc="white", ec="0.35", lw=0.6, alpha=0.94)
AR = dict(arrowstyle="-|>", color="0.95", lw=2.0, shrinkA=1, shrinkB=3, mutation_scale=8)
def lab(t, tx, ty, px, py, fs=8.5):
    ax.annotate(t, xy=(px, py), xytext=(tx, ty), fontsize=fs, ha="center",
                va="center", bbox=BB, arrowprops=AR, zorder=5)

lab("5 V from analyser",        170,  30,  235, 250)
lab("Ethernet \u2192 host PC",  170, 855,  200, 455)
lab("FNB58, 5 V rail",         1820, 855, 1545, 640)
lab("supply in",               1290,  30, 1305, 240)
lab("log \u2192 host PC",       1850, 370, 1805, 215)

fig.tight_layout(pad=0.02)
fig.savefig(OUT_IMG, dpi=300)
plt.show()
print("written:", OUT_IMG)

---

## Appendix

Two diagnostics that are not needed to reproduce the results, but that document
design decisions a reader might otherwise question. Neither writes anything.

### A.1 Why int2 is not in the experiment matrix

A 2-bit symmetric quantizer with a max-based scale has `n = 2^(2-1) - 1 = 1`, so the
scale becomes `max|w|` and `round(w / max|w|)` zeroes every weight below half the
channel maximum. On a typical weight distribution that is roughly 95% of them: the
result is aggressive pruning, not 2-bit quantization, and it collapsed the model in
early runs.

It is also redundant in hardware. The int2 codebook is `{-1, 0, +1}`, identical to
ternary, and both map to the same multiplier-free sign-and-add path. They are not
separate points on the resource axis, so ternary alone represents that arithmetic.

The cell below reproduces the observation.

In [ ]:
# --- A.1 fraction of conv2 weights driven to zero by each mode ---
w2 = ref.get_layer("conv2").get_weights()[0]

def _int2(w):
    """The max-based 2-bit rule, kept here only for this diagnostic."""
    ra = tuple(range(w.ndim - 1))
    s = np.abs(w).max(axis=ra, keepdims=True) / 1.0 + 1e-12
    return np.clip(np.round(w / s), -1, 1) * s

print(f"{'mode':10s}{'zeros':>8s}{'levels in channel 0':>22s}")
for m in ["int8", "int4", "ternary"]:
    q = quant_weight(tf.constant(w2), m).numpy()
    print(f"{m:10s}{100*(q==0).mean():7.1f}%{len(np.unique(q[:,:,0])):22d}")
q = _int2(w2)
print(f"{'int2':10s}{100*(q==0).mean():7.1f}%{len(np.unique(q[:,:,0])):22d}   <- pruning, not quantization")

### A.2 How the number of fine-tuning epochs was fixed

Fine-tuning helps the ternary configuration and hurts the others, and the validation
split cannot tell us where to stop because it is saturated at 0.99. The sweep below
runs 0 to 15 epochs without early stopping and reports both splits.

The pattern it revealed: for `W_int8_A8` the entire clipping benefit is undone in the
**first** epoch and then stays flat, which is domain over-fitting to the training
loads. `W_ternary_A8` climbs to a plateau at three epochs and falls again by fifteen.
Three epochs, fixed and identical for every configuration, sits at that plateau.

This is stated plainly because the choice was made by looking at the shifted-domain
score; a fixed value applied uniformly is more defensible than per-configuration
tuning, but it is not a blind choice.

In [ ]:
# --- A.2 fine-tuning epoch sweep (slow: roughly 5 minutes) ---
PROBE   = ["W_int8_A8", "W_ternary_A8"]
EP_GRID = [0, 1, 2, 3, 5, 8, 15]

sweep = []
for name in PROBE:
    cfg = EXPERIMENTS[name]
    for ep in EP_GRID:
        if ep == 0:
            qm = prepare(cfg)
        else:
            qm = qat_finetune(cfg, seed=SEED, epochs=ep)
        sweep.append(dict(config=name, epoch=ep,
                          val_F1 = q_macro_f1(qm, X_val,  y_val),
                          test_F1= q_macro_f1(qm, X_test, y_test)))

S = pd.DataFrame(sweep)
for name in PROBE:
    s = S[S.config == name]
    print(f"\n--- {name} ---")
    print(f"{'epoch':>6s}{'val_F1':>9s}{'test_F1':>9s}{'val-test':>10s}")
    for _, r in s.iterrows():
        print(f"{int(r.epoch):6d}{r.val_F1:9.4f}{r.test_F1:9.4f}{r.val_F1-r.test_F1:10.4f}")
S.to_csv(os.path.join(DIRS["output"], "qat_epoch_sweep.csv"), index=False)